# Chapter 8 – Unsupervised Learning

이번 실습에서는 비지도학습 중 다음 두 가지 흐름만 다룹니다.

1. **K-Means Clustering**
   - 라벨이 없는 데이터를 비슷한 샘플끼리 묶는 방법
   - 중심점, 군집 번호, 거리, inertia, silhouette score 확인
   - K-Means가 잘 작동하지 않는 경우 확인

2. **Anomaly Detection**
   - Gaussian Mixture Model을 사용해 데이터 밀도를 추정
   - 밀도가 낮은 샘플을 이상치로 판단

### `#TODO` 또는 `____` 부분에 코드를 채워 넣으세요

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/ageron/handson-mlp/blob/main/08_unsupervised_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/ageron/handson-mlp/blob/main/08_unsupervised_learning.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

# Setup

실습에 필요한 기본 라이브러리와 그래프 설정을 준비합니다.

In [ ]:
# 파이썬 버전이 예제 실행 조건을 만족하는지 확인합니다.
import sys

assert sys.version_info >= (3, 10)

In [ ]:
# scikit-learn 버전이 예제 실행 조건을 만족하는지 확인합니다.
from packaging.version import Version
import sklearn

assert Version(sklearn.__version__) >= Version("1.6.1")

In [ ]:
# 수치 계산과 시각화에 사용할 기본 라이브러리를 불러옵니다.
import numpy as np
import matplotlib.pyplot as plt

# 그래프의 글자 크기를 보기 좋게 설정합니다.
plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

# 결과 재현을 위해 난수 시드를 고정합니다.
np.random.seed(42)

# K-Means Clustering

1. 군집 중심점인 **centroid**를 정합니다.
2. 각 샘플을 가장 가까운 centroid에 배정합니다.
3. 각 군집에 속한 샘플들의 평균으로 centroid를 다시 계산합니다.
4. centroid가 거의 움직이지 않을 때까지 반복합니다.

## K-Means 실습용 데이터 만들기

먼저 군집 구조가 비교적 뚜렷한 2차원 데이터를 만듭니다.  
`make_blobs()`는 군집 실습용 가상 데이터를 생성해주는 함수입니다.

In [ ]:
# KMeans: K-Means 모델
# make_blobs: 군집이 있는 가상 데이터를 만들어주는 함수
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

# 각 군집의 중심 위치를 직접 지정합니다.
# 실제 정답 라벨이 없어도 K-Means가 이 구조를 찾아낼 수 있는지 확인할 것입니다.
blob_centers = np.array([
    [ 0.2,  2.3],
    [-1.5,  2.3],
    [-2.8,  1.8],
    [-2.8,  2.8],
    [-2.8,  1.3]
])

# 각 군집의 퍼짐 정도입니다.
# 값이 작을수록 점들이 중심 가까이에 모입니다.
blob_std = np.array([0.4, 0.3, 0.1, 0.1, 0.1])

# X_blobs: 입력 데이터
# y_blobs: make_blobs가 생성한 정답 군집 번호
# 비지도학습에서는 y_blobs를 학습에 사용하지 않습니다.
X_blobs, y_blobs = make_blobs(
    n_samples=2000,
    centers=blob_centers,
    cluster_std=blob_std,
    random_state=7
)

print("X_blobs shape:", X_blobs.shape)
print("y_blobs shape:", y_blobs.shape)

In [ ]:
# 데이터를 산점도로 시각화하는 함수입니다.
# y가 주어지면 군집 또는 클래스별로 색을 다르게 표시할 수 있습니다.
def plot_clusters(X, y=None):
    plt.scatter(X[:, 0], X[:, 1], c=y, s=3)
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$", rotation=0)
    plt.grid()

plt.figure(figsize=(8, 4))
plot_clusters(X_blobs)
plt.title("Generated blob data")
plt.show()

## K-Means 모델 학습

`KMeans`의 가장 중요한 하이퍼파라미터는 `n_clusters`입니다.  
이는 데이터를 몇 개의 군집으로 나눌지 지정합니다.

아래 셀에서는 `k=5`로 두고 K-Means를 학습합니다.

In [ ]:
# k는 만들 군집의 개수입니다.
k = 5

#TODO
# KMeans 모델을 생성하고, X_blobs 데이터에 대해 fit_predict()를 수행하세요.

kmeans = KMeans(n_clusters=____, random_state=42)
y_pred = kmeans.fit_predict(____)

print("처음 10개 샘플의 군집 번호:", y_pred[:10])

In [ ]:
# K-Means가 각 샘플에 부여한 군집 번호입니다.
# 주의: 여기서 labels_는 정답 라벨이 아니라, 모델이 찾은 군집 번호입니다.
kmeans.labels_

In [ ]:
# 각 군집의 중심점입니다.
# 데이터가 2차원이므로 각 centroid도 [x1, x2] 형태입니다.
kmeans.cluster_centers_

## 새로운 데이터의 군집 예측

학습이 끝난 K-Means 모델은 새로운 샘플이 어느 군집에 가까운지도 예측할 수 있습니다.

In [ ]:
# 새롭게 들어온 4개의 샘플입니다.
X_new = np.array([
    [0, 2],
    [3, 2],
    [-3, 3],
    [-3, 2.5]
])

#TODO
# 새 샘플 X_new가 어느 군집에 속하는지 예측하세요.
new_labels = kmeans.predict(____)

print("새 샘플의 군집 번호:", new_labels)

## Hard Clustering과 Soft Clustering

K-Means의 `predict()`는 각 샘플을 가장 가까운 하나의 군집에 배정합니다.  
이처럼 하나의 군집 번호만 주는 방식을 **hard clustering**이라고 합니다.

반면 `transform()`을 사용하면 각 샘플이 모든 centroid에서 얼마나 떨어져 있는지 확인할 수 있습니다.  
이 거리 정보를 보면, 어느 군집에 더 가까운지 더 자세히 이해할 수 있습니다.

In [ ]:
# transform()은 각 샘플과 모든 centroid 사이의 거리를 계산합니다.
# 결과 행: 샘플
# 결과 열: 각 군집 중심점까지의 거리

#TODO
distances = kmeans.transform(____)

np.round(distances, 2)

## K-Means 결정 경계 시각화

아래 함수는 2차원 평면 전체를 촘촘한 격자점으로 나눈 뒤,  
각 격자점이 어느 군집으로 예측되는지 색으로 표시합니다.

이 셀은 시각화를 위한 보조 코드이므로 내부 구현을 모두 외우지 않아도 됩니다.

In [ ]:
# 데이터 점을 표시하는 함수입니다.
def plot_data(X):
    plt.plot(X[:, 0], X[:, 1], 'k.', markersize=2)

# centroid를 동그라미와 x 표시로 강조하는 함수입니다.
def plot_centroids(centroids):
    plt.scatter(
        centroids[:, 0], centroids[:, 1],
        marker='o', s=35, linewidths=8,
        color='w', zorder=10, alpha=0.9
    )
    plt.scatter(
        centroids[:, 0], centroids[:, 1],
        marker='x', s=2, linewidths=12,
        color='k', zorder=11, alpha=1
    )

# K-Means가 만든 결정 경계를 시각화하는 함수입니다.
def plot_decision_boundaries(clusterer, X, resolution=500,
                             show_centroids=True,
                             show_xlabels=True,
                             show_ylabels=True):
    # 데이터 범위보다 약간 넓게 그래프 영역을 잡습니다.
    mins = X.min(axis=0) - 0.1
    maxs = X.max(axis=0) + 0.1

    # 2차원 평면에 격자 좌표를 만듭니다.
    xx, yy = np.meshgrid(
        np.linspace(mins[0], maxs[0], resolution),
        np.linspace(mins[1], maxs[1], resolution)
    )

    # 모든 격자점에 대해 군집 번호를 예측합니다.
    Z = clusterer.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # 예측된 군집 번호에 따라 배경을 칠합니다.
    plt.contourf(
        Z,
        extent=(mins[0], maxs[0], mins[1], maxs[1]),
        cmap="Pastel2"
    )
    plt.contour(
        Z,
        extent=(mins[0], maxs[0], mins[1], maxs[1]),
        linewidths=1,
        colors='k'
    )

    # 실제 데이터를 검은 점으로 표시합니다.
    plot_data(X)

    # centroid를 표시합니다.
    if show_centroids:
        plot_centroids(clusterer.cluster_centers_)

    if show_xlabels:
        plt.xlabel("$x_1$")
    else:
        plt.tick_params(labelbottom=False)

    if show_ylabels:
        plt.ylabel("$x_2$", rotation=0)
    else:
        plt.tick_params(labelleft=False)

In [ ]:
# 학습된 K-Means 모델의 결정 경계를 시각화합니다.
plt.figure(figsize=(8, 4))
plot_decision_boundaries(kmeans, X_blobs)
plt.title("K-Means decision boundaries")
plt.show()

## K-Means 알고리즘의 반복 과정

아래 코드는 K-Means가 반복 횟수에 따라 centroid를 어떻게 움직이는지 보여줍니다.

- `init="random"`: centroid를 랜덤하게 초기화합니다.
- `n_init=1`: 초기화를 한 번만 수행합니다.
- `max_iter`: centroid 업데이트를 몇 번 반복할지 정합니다.

실무에서는 보통 기본값을 사용하지만, 여기서는 알고리즘의 작동 과정을 보기 위해 일부러 반복 횟수를 제한합니다.

In [ ]:
# 반복 횟수에 따라 K-Means 결과가 어떻게 달라지는지 비교합니다.
# max_iter=1, 2, 3으로 설정해 centroid가 점점 이동하는 과정을 확인합니다.

kmeans_iter1 = KMeans(
    n_clusters=5,
    init="random",
    n_init=1,
    max_iter=1,
    random_state=18
)

kmeans_iter2 = KMeans(
    n_clusters=5,
    init="random",
    n_init=1,
    max_iter=2,
    random_state=18
)

kmeans_iter3 = KMeans(
    n_clusters=5,
    init="random",
    n_init=1,
    max_iter=3,
    random_state=18
)

kmeans_iter1.fit(X_blobs)
kmeans_iter2.fit(X_blobs)
kmeans_iter3.fit(X_blobs)

In [ ]:
# 반복 횟수별 결정 경계를 비교합니다.
fig, axes = plt.subplots(ncols=3, figsize=(15, 4), sharey=True)

for ax, model, title in zip(
    axes,
    [kmeans_iter1, kmeans_iter2, kmeans_iter3],
    ["max_iter=1", "max_iter=2", "max_iter=3"]
):
    plt.sca(ax)
    plot_decision_boundaries(model, X_blobs, resolution=300,
                             show_ylabels=(ax == axes[0]))
    plt.title(title)

plt.show()

## Inertia

K-Means는 정답 라벨이 없는 비지도학습이므로 일반적인 정확도 계산이 어렵습니다.  
대신 K-Means는 각 샘플이 자신이 속한 centroid에서 얼마나 가까운지를 사용합니다.

**Inertia**는 각 샘플과 가장 가까운 centroid 사이의 거리 제곱합입니다.

- inertia가 작을수록 샘플들이 centroid에 가깝습니다.
- 하지만 `k`를 계속 키우면 inertia는 거의 항상 감소합니다.
- 따라서 inertia만 보고 무조건 가장 작은 값을 고르면 안 됩니다.

In [ ]:
#TODO
# 학습된 K-Means 모델의 inertia_ 값을 확인하세요.
inertia = kmeans.____

print("Inertia:", inertia)

# score()는 음수 inertia를 반환합니다.
# scikit-learn의 score()는 일반적으로 클수록 좋다는 규칙을 따르기 때문입니다.
print("KMeans score:", kmeans.score(X_blobs))

## 여러 k 값 비교하기: Elbow Method

군집 개수 `k`를 1부터 9까지 바꿔가며 K-Means를 학습하고,  
각 모델의 inertia를 비교합니다.

inertia 그래프에서 감소폭이 갑자기 완만해지는 지점을 **elbow**라고 부릅니다.

In [ ]:
#TODO
# k=1부터 k=9까지 K-Means 모델을 학습하세요.

kmeans_per_k = [
    KMeans(n_clusters=____, random_state=42).fit(____)
    for k in range(____, ____)
]

# 각 모델의 inertia를 리스트로 저장합니다.
inertias = [model.____ for model in kmeans_per_k]

inertias

In [ ]:
# k에 따른 inertia 변화를 그래프로 확인합니다.
plt.figure(figsize=(8, 4))
plt.plot(range(1, 10), inertias, "bo-")
plt.xlabel("$k$")
plt.ylabel("Inertia")
plt.title("Elbow method")
plt.grid()
plt.show()

## Silhouette Score

silhouette score는 각 샘플이 자기 군집 안에 잘 들어가 있고,  
다른 군집과는 충분히 떨어져 있는지를 측정합니다.

값의 범위는 대략 다음과 같이 해석할 수 있습니다.

- `+1`에 가까움: 군집이 잘 나뉨
- `0`에 가까움: 군집 경계에 있음
- `-1`에 가까움: 잘못된 군집에 배정되었을 가능성이 큼

In [ ]:
# silhouette_score는 군집 결과의 품질을 평가하는 함수입니다.
from sklearn.metrics import silhouette_score

#TODO
# kmeans 모델의 silhouette score를 계산하세요.
score = silhouette_score(____, ____)

print("Silhouette score:", score)

In [ ]:
# k=2부터 k=9까지 silhouette score를 계산합니다.
# k=1은 군집이 하나뿐이므로 silhouette score를 계산할 수 없습니다.

#TODO
silhouette_scores = [
    silhouette_score(X_blobs, model.____)
    for model in kmeans_per_k[1:]
]

plt.figure(figsize=(8, 4))
plt.plot(range(2, 10), silhouette_scores, "bo-")
plt.xlabel("$k$")
plt.ylabel("Silhouette score")
plt.title("Silhouette scores by k")
plt.grid()
plt.show()

## Mini-Batch K-Means

데이터가 매우 클 때는 전체 데이터를 매번 사용하지 않고,  
작은 mini-batch만 사용해 centroid를 업데이트하는 `MiniBatchKMeans`를 사용할 수 있습니다.

일반 K-Means보다 빠르지만, 결과가 약간 달라질 수 있습니다.

In [ ]:
from sklearn.cluster import MiniBatchKMeans

#TODO
# MiniBatchKMeans를 사용해 같은 데이터를 군집화하세요.
minibatch_kmeans = MiniBatchKMeans(
    n_clusters=____,
    random_state=42
)

minibatch_kmeans.fit(____)

print("MiniBatch K-Means inertia:", minibatch_kmeans.inertia_)

## K-Means의 한계

K-Means는 군집이 둥글고 크기가 비슷할 때 잘 작동합니다.  
하지만 군집이 길쭉하거나 밀도가 서로 다르면 좋은 결과를 내기 어렵습니다.

아래 데이터는 일부러 K-Means가 어려워하는 형태로 만든 데이터입니다.

In [ ]:
# 길쭉한 군집과 크기가 다른 군집을 가진 데이터를 만듭니다.
X1, y1 = make_blobs(
    n_samples=1000,
    centers=((4, -4), (0, 0)),
    random_state=42
)

# 선형 변환을 적용해 둥근 군집을 길쭉한 타원 형태로 바꿉니다.
X1 = X1.dot(np.array([
    [0.374, 0.95],
    [0.732, 0.598]
]))

# 오른쪽 아래에 작은 군집을 추가합니다.
X2, y2 = make_blobs(
    n_samples=250,
    centers=1,
    random_state=42
)
X2 = X2 + [6, -8]

# 두 데이터셋을 하나로 합칩니다.
X_complex = np.r_[X1, X2]
y_complex = np.r_[y1, y2]

plt.figure(figsize=(8, 4))
plot_clusters(X_complex)
plt.title("Difficult data for K-Means")
plt.show()

In [ ]:
#TODO
# 복잡한 데이터 X_complex에 K-Means를 적용하세요.
# 군집 개수는 3개로 설정합니다.

kmeans_complex = KMeans(
    n_clusters=____,
    n_init=10,
    random_state=42
)

kmeans_complex.fit(____)

plt.figure(figsize=(8, 4))
plot_decision_boundaries(kmeans_complex, X_complex, resolution=300)
plt.title(f"K-Means on difficult data | Inertia = {kmeans_complex.inertia_:.1f}")
plt.show()

### 생각해보기

K-Means가 위 데이터에서 완벽하지 않은 이유를 생각해봅시다.

- K-Means는 각 샘플을 가장 가까운 centroid에 배정합니다.
- 이 방식은 군집이 둥글고 크기가 비슷하다는 가정과 잘 맞습니다.
- 길쭉한 군집이나 밀도가 다른 군집에서는 centroid와의 거리만으로는 좋은 군집을 만들기 어렵습니다.

# Anomaly Detection Using Gaussian Mixtures

이번 파트에서는 `GaussianMixture`를 사용해 데이터의 밀도를 추정하고,  
밀도가 낮은 위치에 있는 샘플을 이상치로 판단합니다.

여기서는 앞에서 K-Means가 어려워했던 `X_complex` 데이터를 그대로 사용합니다.

## Gaussian Mixture Model 학습

Gaussian Mixture Model, 줄여서 GMM은 데이터를 여러 개의 가우시안 분포가 섞여 만들어졌다고 가정합니다.

K-Means와의 차이는 다음과 같습니다.

- K-Means: 각 샘플을 가장 가까운 centroid에 배정
- GMM: 각 샘플이 각 가우시안 분포에서 나왔을 확률을 계산
- K-Means: 주로 둥근 군집에 강함
- GMM: 공분산을 사용해 타원형 군집도 표현 가능

In [ ]:
# GaussianMixture: 여러 개의 가우시안 분포가 섞인 모델입니다.
from sklearn.mixture import GaussianMixture

#TODO
# X_complex 데이터에 Gaussian Mixture Model을 학습하세요.
# n_components는 사용할 가우시안 분포의 개수입니다.
# covariance_type="full"은 각 군집이 자유로운 타원형 모양을 가질 수 있게 합니다.

gm = GaussianMixture(
    n_components=____,
    covariance_type="full",
    n_init=10,
    random_state=42
)

gm.fit(____)

In [ ]:
# 모델이 추정한 각 가우시안 분포의 비율입니다.
# 세 값의 합은 1에 가깝습니다.
gm.weights_

In [ ]:
# 각 가우시안 분포의 평균 위치입니다.
# K-Means의 cluster_centers_와 비슷하게 볼 수 있지만,
# GMM에서는 확률 분포의 평균이라는 의미입니다.
gm.means_

In [ ]:
# 각 가우시안 분포의 공분산 행렬입니다.
# 공분산은 분포의 퍼짐과 기울어진 방향을 나타냅니다.
gm.covariances_

In [ ]:
# GMM은 EM 알고리즘을 반복해서 파라미터를 추정합니다.
# converged_가 True이면 알고리즘이 수렴했다는 의미입니다.
print("수렴 여부:", gm.converged_)
print("반복 횟수:", gm.n_iter_)

## GMM으로 군집 확률 확인하기

`predict()`는 가장 가능성이 높은 군집 하나를 반환합니다.  
`predict_proba()`는 각 샘플이 각 군집에 속할 확률을 반환합니다.

In [ ]:
# hard clustering: 가장 가능성이 높은 군집 번호를 반환합니다.
gmm_labels = gm.predict(X_complex)

print("처음 10개 샘플의 GMM 군집 번호:")
print(gmm_labels[:10])

In [ ]:
# soft clustering: 각 군집에 속할 확률을 반환합니다.
# 행은 샘플, 열은 가우시안 성분입니다.
gmm_probs = gm.predict_proba(X_complex)

np.round(gmm_probs[:10], 3)

## GMM 밀도 시각화 함수

아래 함수는 GMM이 추정한 밀도와 군집 경계를 시각화합니다.

- 배경의 등고선: 밀도 수준
- 점선 경계: GMM이 예측한 군집 경계
- 검은 점: 실제 데이터

In [ ]:
from matplotlib.colors import LogNorm

# GMM의 밀도와 군집 경계를 그리는 함수입니다.
def plot_gaussian_mixture(clusterer, X, resolution=500, show_ylabels=True):
    # 데이터 범위보다 약간 넓게 그래프 영역을 잡습니다.
    mins = X.min(axis=0) - 0.1
    maxs = X.max(axis=0) + 0.1

    # 2차원 격자 좌표를 만듭니다.
    xx, yy = np.meshgrid(
        np.linspace(mins[0], maxs[0], resolution),
        np.linspace(mins[1], maxs[1], resolution)
    )

    # score_samples()는 log density를 반환합니다.
    # 여기서는 시각화를 위해 음수 값을 사용해 등고선을 그립니다.
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    Z = -clusterer.score_samples(grid_points)
    Z = Z.reshape(xx.shape)

    # 밀도 등고선을 그립니다.
    plt.contourf(
        xx, yy, Z,
        norm=LogNorm(vmin=1.0, vmax=30.0),
        levels=np.logspace(0, 2, 12)
    )
    plt.contour(
        xx, yy, Z,
        norm=LogNorm(vmin=1.0, vmax=30.0),
        levels=np.logspace(0, 2, 12),
        linewidths=1,
        colors='k'
    )

    # 각 격자점의 군집 예측값을 이용해 결정 경계를 그립니다.
    Z_labels = clusterer.predict(grid_points)
    Z_labels = Z_labels.reshape(xx.shape)
    plt.contour(
        xx, yy, Z_labels,
        linewidths=2,
        colors='r',
        linestyles='dashed'
    )

    # 실제 데이터를 표시합니다.
    plt.plot(X[:, 0], X[:, 1], 'k.', markersize=2)

    # GMM 평균 위치를 표시합니다.
    plot_centroids(clusterer.means_)

    plt.xlabel("$x_1$")
    if show_ylabels:
        plt.ylabel("$x_2$", rotation=0)
    else:
        plt.tick_params(labelleft=False)

In [ ]:
plt.figure(figsize=(8, 4))
plot_gaussian_mixture(gm, X_complex)
plt.title("Gaussian Mixture density and cluster boundaries")
plt.show()

## 이상치 탐지

GMM의 `score_samples()`는 각 샘플 위치의 **log density**를 반환합니다.

- density가 높음: 모델이 보기에는 흔한 위치
- density가 낮음: 모델이 보기에는 드문 위치
- 밀도가 매우 낮은 샘플: 이상치 후보

여기서는 밀도가 가장 낮은 하위 2% 샘플을 이상치로 표시합니다.

In [ ]:
#TODO
# 각 샘플의 log density를 계산하세요.

densities = gm.____(____)

# 하위 2%에 해당하는 density 값을 임계값으로 사용합니다.
# 즉, 전체 샘플 중 약 2%를 이상치 후보로 봅니다.
density_threshold = np.percentile(densities, ____)

# density가 임계값보다 낮은 샘플을 이상치로 선택합니다.
anomalies = X_complex[densities < ____]

print("density threshold:", density_threshold)
print("이상치 후보 개수:", len(anomalies))

In [ ]:
# 이상치를 별표로 표시합니다.
plt.figure(figsize=(8, 4))
plot_gaussian_mixture(gm, X_complex, resolution=500)

plt.scatter(
    anomalies[:, 0],
    anomalies[:, 1],
    color='r',
    marker='*',
    s=80,
    label="Anomaly"
)

plt.title("Anomaly detection using Gaussian Mixture")
plt.legend()
plt.show()

## 이상치 비율 바꿔보기

실제 문제에서는 이상치 비율을 사전에 알고 있거나,  
업무 상황에 맞게 조정해야 하는 경우가 많습니다.

아래 셀에서 `percentile` 값을 바꾸면서 이상치 개수가 어떻게 달라지는지 확인해보세요.

In [ ]:
#TODO
# percentile 값을 1, 2, 5, 10 등으로 바꿔보세요.
percentile = ____

threshold = np.percentile(densities, percentile)
anomalies_custom = X_complex[densities < threshold]

print(f"하위 {percentile}% 기준 이상치 개수:", len(anomalies_custom))

plt.figure(figsize=(8, 4))
plot_gaussian_mixture(gm, X_complex, resolution=500)
plt.scatter(
    anomalies_custom[:, 0],
    anomalies_custom[:, 1],
    color='r',
    marker='*',
    s=80,
    label=f"Bottom {percentile}%"
)
plt.title("Anomaly detection with custom percentile")
plt.legend()
plt.show()

# 정리

이번 실습에서 확인한 내용은 다음과 같습니다.

1. K-Means는 centroid를 기준으로 데이터를 `k`개의 군집으로 나눕니다.
2. `fit_predict()`는 학습과 군집 번호 예측을 동시에 수행합니다.
3. `cluster_centers_`는 각 군집의 중심점입니다.
4. `inertia_`는 샘플과 centroid 사이의 거리 제곱합입니다.
5. `k`를 고를 때는 elbow method와 silhouette score를 함께 볼 수 있습니다.
6. K-Means는 길쭉하거나 밀도가 다른 군집에서 한계가 있습니다.
7. Gaussian Mixture Model은 데이터 밀도를 추정할 수 있습니다.
8. 밀도가 낮은 샘플은 이상치 후보로 볼 수 있습니다.

생각해볼 질문:

- K-Means에서 `k`를 잘못 고르면 어떤 문제가 생길까요?
- inertia와 silhouette score가 서로 다른 결론을 줄 수도 있을까요?
- 이상치 비율을 2%로 정하는 것은 항상 합리적일까요?